In [2]:
import sys, os
import cv2, numpy as np, matplotlib.pyplot as plt

# Paksa sistem buat nambahin folder src langsung dari path Google Drive kamu
sys.path.append('/content/drive/MyDrive/simuteh-2330511031/src')
from utils_cv import baca, tampil, kontras_bercak, statistik

# Pindah ke folder notebook agar pemanggilan path ../data berfungsi
os.chdir('/content/drive/MyDrive/simuteh-2330511031/notebook')

# ---------------- L3.2: Diagnosis Mutu (Statistik & Histogram) ----------------
print("--- L3.2: Statistik 5 Kondisi (Kelas Berat) ---")
plt.figure(figsize=(10, 5))
for nama in ["teh_041.png", "teh_042.png", "teh_043.png", "teh_044.png", "teh_045.png"]:
    gray = cv2.cvtColor(baca(f"../data/sintetis/citra/{nama}"), cv2.COLOR_BGR2GRAY)
    print(f"{nama}: {statistik(gray)}")
    hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
    plt.plot(hist, label=nama)
plt.xlabel("Intensitas"); plt.ylabel("Jumlah piksel"); plt.legend()
plt.savefig("../hasil/b3_histogram_5kondisi.png")
plt.show()

print("\n--- L3.2: Statistik 5 Kondisi (Kelas Ringan) ---")
for nama in ["teh_021.png", "teh_022.png", "teh_023.png", "teh_024.png", "teh_025.png"]:
    gray = cv2.cvtColor(baca(f"../data/sintetis/citra/{nama}"), cv2.COLOR_BGR2GRAY)
    print(f"{nama}: {statistik(gray)}")


# ---------------- L3.3: Perbaikan Citra (Bayangan) ----------------
print("\n--- L3.3: Perbaikan Citra Kondisi Bayangan (teh_045.png) ---")
nama = "teh_045.png"
img = baca(f"../data/sintetis/citra/{nama}")
md = cv2.imread(f"../data/sintetis/mask_daun/{nama}", 0)
mb = cv2.imread(f"../data/sintetis/mask_bercak/{nama}", 0)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

def gamma_lut(g):
    lut = np.array([255 * (i / 255) ** g for i in range(256)])
    return lut.clip(0, 255).astype(np.uint8)

rmin, rmax = np.percentile(gray, (2, 98))
hasil = {
    "asli": gray,
    "negatif": 255 - gray,
    "cerah+50": cv2.add(gray, 50),
    "kontras": cv2.convertScaleAbs(gray, alpha=1.5, beta=-40),
    "regang": np.clip((gray - rmin) / (rmax - rmin) * 255, 0, 255).astype(np.uint8),
    "gamma0.5": cv2.LUT(gray, gamma_lut(0.5)),
    "gamma1.8": cv2.LUT(gray, gamma_lut(1.8)),
    "ekualisasi": cv2.equalizeHist(gray),
    "CLAHE": cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray),
}

for nm, c in hasil.items():
    print(f"{nm:10s} KB={kontras_bercak(c, md, mb):.2f} sigma={c.std():.1f}")


# ---------------- L3.4: Eksperimen CLAHE & Citra Warna ----------------
print("\n--- L3.4: Eksperimen Parameter CLAHE ---")
for clip in [1.0, 2.0, 4.0]:
    for tile in [4, 8, 16]:
        c = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile)).apply(gray)
        print(f"clip={clip} tile={tile}: KB={kontras_bercak(c, md, mb):.2f}")

clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
hsv[:, :, 2] = clahe.apply(hsv[:, :, 2])
via_v = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
via_bgr = cv2.merge([clahe.apply(c) for c in cv2.split(img)])
tampil([img, via_v, via_bgr], ["Asli (Bayangan)", "CLAHE kanal V (Benar)", "CLAHE tiap kanal BGR (Kacau)"])


# ---------------- L3.5: Perbaikan Mandiri (Gelap, Terang, KR, Bayangan) ----------------
print("\n--- L3.5: Uji Coba Perbaikan Spesifik per Kondisi ---")

def terapkan_perbaikan(nama_file, metode):
    im = baca(f"../data/sintetis/citra/{nama_file}")
    m_d = cv2.imread(f"../data/sintetis/mask_daun/{nama_file}", 0)
    m_b = cv2.imread(f"../data/sintetis/mask_bercak/{nama_file}", 0)
    gr = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)

    if metode == "Gamma 0.5": res = cv2.LUT(gr, gamma_lut(0.5))
    elif metode == "Gamma 1.8": res = cv2.LUT(gr, gamma_lut(1.8))
    elif metode == "Regang":
        p2, p98 = np.percentile(gr, (2, 98))
        res = np.clip((gr - p2) / (p98 - p2) * 255, 0, 255).astype(np.uint8)
    elif metode == "CLAHE":
        res = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gr)

    kb_awal = kontras_bercak(gr, m_d, m_b)
    kb_akhir = kontras_bercak(res, m_d, m_b)
    return gr, res, kb_awal, kb_akhir

skenario = [
    ("Gelap", "Gamma 0.5", "teh_042.png", "teh_022.png"),
    ("Terang", "Gamma 1.8", "teh_043.png", "teh_023.png"),
    ("Kontras Rendah", "Regang", "teh_044.png", "teh_024.png"),
    ("Bayangan", "CLAHE", "teh_045.png", "teh_025.png")
]

for kondisi, met, b_name, r_name in skenario:
    g_b, res_b, kba_b, kbr_b = terapkan_perbaikan(b_name, met)
    g_r, res_r, kba_r, kbr_r = terapkan_perbaikan(r_name, met)

    tampil(
        [g_b, res_b, g_r, res_r],
        [f"{b_name} Asli (KB {kba_b:.2f})", f"{met} (KB {kbr_b:.2f})",
         f"{r_name} Asli (KB {kba_r:.2f})", f"{met} (KB {kbr_r:.2f})"],
        baris=1, ukuran=(16, 4)
    )

Output hidden; open in https://colab.research.google.com to view.